# Geomechanical Injection Scenario Toolkit (GIST)

#Disclaimer
GIST aims to give the _gist_ of a wide range of potential scenarios and aid collective decision making when responding to seismicity.

The results of GIST are entirely dependent upon the inputs provided, which may be incomplete or inaccurate.

There are other potentially plausible inducement scenarios that are not considered, including fluid migration into the basement, 
out-of-zone poroelastic stressing, or hydraulic fracturing.

None of the individual models produced by GIST accurately represent what happens in the subsurface and cannot be credibly used 
to accurately assign liability or responsibility for seismicity.

"All models are wrong, but some are useful" - George Box, 1976

## Prerequisites

Assumes InjectionSQLScheduled completed successfully and injection data are sampled uniformly in time

In [0]:
%run "./GIST_RunTemplate_ErrorReporting_Step1"

# Geomechanical Injection Scenario Toolkit (GIST)

#Disclaimer
GIST aims to give the _gist_ of a wide range of potential scenarios and aid collective decision making when responding to seismicity.

The results of GIST are entirely dependent upon the inputs provided, which may be incomplete or inaccurate.

There are other potentially plausible inducement scenarios that are not considered, including fluid migration into the basement, 
out-of-zone poroelastic stressing, or hydraulic fracturing.

None of the individual models produced by GIST accurately represent what happens in the subsurface and cannot be credibly used 
to accurately assign liability or responsibility for seismicity.

"All models are wrong, but some are useful" - George Box, 1976

## Prerequisites

Assumes InjectionSQLScheduled completed successfully and injection data are sampled uniformly in time

##Install Dependencies
- geopandas
- gistMC.py
- eqSQL.py
- gistPlots.py
- numpy
- scipy
- pandas


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.5/32.5 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 93.2 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 47.7 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


...

##Libraries

- numpy
- scipy
- pandas
- matplotlib
- geopandas
- pyspark


#1. Select Event

##1.1. Parameters: TexNet event ID + forecast duration

##1.2. Results: Create directories, fetch EQ

getEarthquake:     SeismicEventId  ... B3RecordDeletedUTCDateTime
0         2889501  ...                        NaT
1         2889531  ...                        NaT

[2 rows x 32 columns] 2
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 35 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   SeismicEventId              2 non-null      int64         
 1   DataSource                  2 non-null      object        
 2   DataSourceUrl               1 non-null      object        
 3   EventID                     2 non-null      object        
 4   EventTimeUtc                2 non-null      object        
 5   EventTimeInLocalTimeZone    2 non-null      object        
 6   EventTimeZone               2 non-null      object        
 7   EventType                   2 non-null      object        
 8   DepthKm                     2 non-null      float64    

#2. Initial Run

##2.1. Parameters
###2.1.1: Subsurface Modeling

In [0]:
# Binary deep/shallow parameter
deepOrShallow='Deep'
# Depth from surface (ft)
# Distinguishes deep/shallow where we don't have a horizon 
depthCutoff=8000.

In [0]:
nRealizations=250

In [0]:
# Porosity in percent
PorosityPercentMin=3.
PorosityPercentMax=12.

# Permeability in millidarcies
PermMDMin=20.
PermMDMax=400.

# Thickness in feet
ThicknessFTMin=200.
ThicknessFTMax=500.

# Vertical compressibility minimum/maximum (1/Pa)
VerticalCompressibilityMin=0.00000000100
VerticalCompressibilityMax=0.00000000110

In [0]:
# Water density minimum/maximum (kg/m3)
WaterDensityMin=1000.
WaterDensityMax=1050.
# Water viscosity minimum/maximum (Pa.s)
WaterViscosityMin=0.000799
WaterViscosityMax=0.000801
# Fluid Compressibility minimum/maximum (1/Pa)
FluidCompressibilityMin=0.000000000359
FluidCompressibilityMax=0.000000000361

In [0]:
# Oversampling of time axis (poroelastic-only)
nTimeBins=21
# Poroelastic parameters - for in-zone poroelasticity in v2
ShearModulusMin=4e9
ShearModulusMax=6e9
PoissonsRatioDrainedMin=0.295
PoissonsRatioDrainedMax=0.305
PoissonsRatioUndrainedMin=0.305
PoissonsRatioUndrainedMax=0.315
BiotsCoefficientMin=0.26
BiotsCoefficientMax=0.36
# Fault parameters (poroelastic-only)
FaultFrictionCoeffMin=0.55
FaultFrictionCoeffMax=0.65
RockFrictionCoeffMin=0.55
RockFrictionCoeffMax=0.65

###2.1.2: Output Filtering

In [0]:
# Verbosity - 0=silent, 1=some, 2=lots
verb=0
# Minimum Pressure change to care about in PSI
dPCutoff=1.
# Maximum Number of wells to plot
nWells=30
# What is the first year we plot?
minYear=1980

##2.2. Results:
### 2.2.1: Set Interval-Specific Paths


In [0]:
# Set an output directory for this earthquake and this interval
runIntervalPath=runPath+deepOrShallow+'/'
initialRunIntervalPath=runIntervalPath+'initialRun/'
updatedRunIntervalPath=runIntervalPath+'udpatedRun/'
forecastRunIntervalPath=runIntervalPath+'forecastRun/'
disposalPath=runIntervalPath+'updatedDisposal/'
# Make directory if it doesn't exist
os.makedirs(runIntervalPath, exist_ok=True)
os.makedirs(initialRunIntervalPath, exist_ok=True)
os.makedirs(updatedRunIntervalPath, exist_ok=True)
os.makedirs(forecastRunIntervalPath, exist_ok=True)
os.makedirs(disposalPath, exist_ok=True)
os.makedirs(initialRunIntervalPath+'perWell', exist_ok=True)
os.makedirs(updatedRunIntervalPath+'perWell', exist_ok=True)
os.makedirs(forecastRunIntervalPath+'perWell', exist_ok=True)
################################################
# Output prefix for realizations of parameters #
################################################
RealizationPrefix=runIntervalPath+'MC'

In [0]:
# Point to appropriate well and injection files from initial database
if deepOrShallow=='Deep':
  WellFile=injPath+'/deep.csv'
  InjFile=injPath+'/deepReg.csv'
elif deepOrShallow=='Shallow':
  WellFile=injPath+'/shallow.csv'
  InjFile=injPath+'/shallowReg.csv'

###2.2.2: Initialize and Select Wells

In [0]:
gist=gi.gistMC(nReal=nRealizations,
                ntBin=nTimeBins)
gist.initPP(rho0_min=WaterDensityMin,
             rho0_max=WaterDensityMax,
             phi_min=PorosityPercentMin,
             phi_max=PorosityPercentMax,
             kMD_min=PermMDMin,
             kMD_max=PermMDMax,
             h_min=ThicknessFTMin,
             h_max=ThicknessFTMax,
             alphav_min=VerticalCompressibilityMin,
             alphav_max=VerticalCompressibilityMax,
             beta_min=FluidCompressibilityMin,
             beta_max=FluidCompressibilityMax)

In [0]:
gist.addWells(wellFile=WellFile,injFile=InjFile,verbose=verb)
if verb>0:
  gist.wellDF.info()
  EQDF.info()

/Workspace/Users/bill.curry@exxonmobil.com/GIST/lib/gistMC.py:728: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  injWellDayMax=float(injWellDays.max())
/Workspace/Users/bill.curry@exxonmobil.com/GIST/lib/gistMC.py:730: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  self.injDT=float(injWellDDay)
/Workspace/Users/bill.curry@exxonmobil.com/GIST/lib/gistMC.py:731: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  self.injOT=float(injWellDayMin)
/Workspace/Users/bill.curry@exxonmobil.com/GIST/lib/gistMC.py:732: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  self.injNT=1+int((injWellDayMax-injWellDayMin)/injWellD

In [0]:
print(eq.info)
print(EQDF.info())
eqDict=eq.to_dict()

<bound method Series.info of SeismicEventId                                     2889501
DataSource                       TexNet Earthquake Catalog
DataSourceUrl                                         None
EventID                                       tx2025zqwqjm
EventTimeUtc                          2025-12-30T07:03:26Z
EventTimeInLocalTimeZone              2025-12-30T01:03:26Z
EventTimeZone                                          CST
EventType                                       Earthquake
DepthKm                                           8.837891
DepthErrorKm                                      1.028479
Magnitude                                         2.518896
MagnitudeError                                         NaN
MagnitudeType                                   ml(texnet)
Location                                     Western Texas
Status                                               final
Latitude                                         31.946411
LatitudeError              

In [0]:
selectedWellsDF,ignoredWellsDF,injDF=gist.findWellsVec(eqDict,PE=False,responseYears=forecastYears,verbose=verb)
# Isolate wells included only for forecast
currentWellsDF=selectedWellsDF[selectedWellsDF['EncompassingDay']<0.].reset_index(drop=True)
forecastWellsDF=selectedWellsDF[selectedWellsDF['EncompassingDay']>=0.].reset_index(drop=True)
print(selectedWellsDF.shape,currentWellsDF.shape,forecastWellsDF.shape)

0      2097-06-13 00:00:00.000000000
1      2069-12-07 00:00:00.000000000
2      2091-01-02 00:00:00.000000000
3      2052-06-27 00:00:00.000000000
4      2101-03-23 00:00:00.000000000
                    ...             
2864   2068-03-18 20:46:12.578921561
2865   2069-12-07 00:00:00.000000000
2866   2113-01-18 00:00:00.000000000
2867   2019-09-20 18:27:03.971703913
2868   2038-10-10 07:09:59.666877748
Length: 2869, dtype: datetime64[ns] 0       26098
1       16048
2       23744
3        9676
4       27476
        ...  
2864    15419
2865    16048
2866    31795
2867    -2293
2868     4667
Length: 2869, dtype: int64
(227, 70) (215, 70) (12, 70)


In [0]:
print(selectedWellsDF.columns)
print(currentWellsDF.columns)

Index(['ID', 'InjectionWellId', 'UniqueWellIdentifier', 'UICNumber',
       'APINumber', 'LeaseName', 'Operator', 'OperatorType',
       'OperatorPrincipalCompany', 'OperatorPrincipalCompanyType', 'WellName',
       'WellNumber', 'State', 'Basin', 'County', 'District', 'SRAOrSIR',
       'B3InjectionType', 'B3InjectionStatus', 'RegulatoryInjectionType',
       'PermittedMaxLiquidBPD', 'PermittedMaxLiquidPSIG',
       'PermittedMaxGasMCFPerDay', 'PermittedMaxGasPSIG',
       'PermittedCommercialStatus', 'PermittedIntervalTopFt',
       'PermittedIntervalBottomFt', 'InjectionClass', 'PermitStage',
       'PermittedWellDepthClassification', 'DaysApplicationHasBeenInReview',
       'DaysToPermitApproval', 'PermitIsAmendment',
       'PendingApplicationIsAmendment',
       'PendingApplicationRequestedCommercialStatus',
       'PendingApplicationRequestedIntervalTopFt',
       'PendingApplicationRequestedIntervalBottomFt',
       'PendingApplicationRequestedMaxLiquidBPD',
       'PendingAppl

###2.2.3: Calculate Pressure Ranges of Selected Wells

In [0]:
scenarioDF=gist.runPressureScenariosVec(eqDict,currentWellsDF,injDF,verbose=1)

/Workspace/Users/bill.curry@exxonmobil.com/GIST/lib/gistMC.py:3017: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  allScenariosDF=pd.concat([allScenariosDF,scenarioDF],ignore_index=True)


In [0]:
filteredDF,orderedWellIDList=gi.summarizePPResults(scenarioDF,currentWellsDF,threshold=dPCutoff,nOrder=nWells,verbose=2)

4  disaggregationPlotPP: wells have a  1.0  psi pressure contribution in one scenario
 disaggregationPlotPP:  4  sorted
 disaggregationPlotPP:  211  minimally-contributing wells sorted


###2.2.4: Generate selection and disaggregation plots

In [0]:
disaggregationDF=gi.prepDisaggregationPlot(filteredDF,orderedWellIDList,jitter=0.1,verbose=0)

/Workspace/Users/bill.curry@exxonmobil.com/GIST/lib/gistMC.py:3967: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  disaggregationPlotDF=pd.concat([disaggregationPlotDF,wellDF],ignore_index=True)


In [0]:
diffRange=(min(gist.diffPPVec),max(gist.diffPPVec))
rtDF,mergedWellsDF = gi.prepRTPlot(selectedWellsDF,ignoredWellsDF,minYear,diffRange,eq,clipYear=False,verbose=1)

 prepRTPlot: years before earthquake to plot: -45.9958932238193
 prepRTPlot: # of must include wells : 6
 prepRTPlot: # of wells to add to forecast : 12
 prepRTPlot: # of wells with 0 BBL disposal : 132


###2.2.5: Generate Time Series Pressures for significant wells

In [0]:
winWellsDF,winInjDF=gi.getWinWells(filteredDF,currentWellsDF,injDF)
display(winWellsDF)

index,ID,InjectionWellId,UniqueWellIdentifier,UICNumber,APINumber,LeaseName,Operator,OperatorType,OperatorPrincipalCompany,OperatorPrincipalCompanyType,WellName,WellNumber,State,Basin,County,District,SRAOrSIR,B3InjectionType,B3InjectionStatus,RegulatoryInjectionType,PermittedMaxLiquidBPD,PermittedMaxLiquidPSIG,PermittedMaxGasMCFPerDay,PermittedMaxGasPSIG,PermittedCommercialStatus,PermittedIntervalTopFt,PermittedIntervalBottomFt,InjectionClass,PermitStage,PermittedWellDepthClassification,DaysApplicationHasBeenInReview,DaysToPermitApproval,PermitIsAmendment,PendingApplicationIsAmendment,PendingApplicationRequestedCommercialStatus,PendingApplicationRequestedIntervalTopFt,PendingApplicationRequestedIntervalBottomFt,PendingApplicationRequestedMaxLiquidBPD,PendingApplicationRequestedMaxLiquidPSIG,PendingApplicationRequestedMaxGasMCFPerDay,PendingApplicationRequestedMaxGasPSIG,PendingApplicationRequestedOperator,PendingApplicationRequestedOperatorType,PendingApplicationRequestedOperatorPrincipalCompany,PendingApplicationRequestedOperatorPrincipalCompanyType,PendingApplicationRequestedWellDepthClassification,CompletedWellDepthClassification,CompletionAndDrillingPermitStatus,StartDate,TotalVerticalDepthFt,MeasuredDepthFt,WellboreOrientation,IsOpenHole,SurfaceHoleElevationFt,SurfaceHoleLatitude,SurfaceHoleLongitude,SurfaceHoleGeographySource,SurveyLinesDescription,B3RecordAddedUTCDateTime,B3RecordUpdatedUTCDateTime,B3RecordDeletedUTCDateTime,Distances,DXs,DYs,DDRatio,YearsInjecting,EncompassingDay,EncompassingDiffusivity,EventID,TotalBBL
97,2111690,111690,10142010000111690,115897,42-173-37815,MARBILL SWD,GOODNIGHT MIDSTREAM,Water Management,GOODNIGHT MIDSTREAM,Water Management,MARBILL SWD 1,1,TX,Permian - Midland,Glasscock,08,null,Saltwater Disposal,Active - Liquid,Injection Into Non-Productive Zone,40000.0,5600.0,0.0,null,Non Commercial,11200.0,12500.0,Class 2 - Injection of fluids associated with oil and natural gas production,Not in Permitting Process,Permian Deep,null,70.0,false,null,null,null,null,null,null,null,null,null,null,null,null,null,Permian Deep,Completed,2018-03-25,12500.0,12000.0,Vertical,true,2594.0,31.98868,-101.698555,Lat/Long from Well Surface Location,2251' FWL & 2225' FSL,2024-02-16 20:41:47,2025-11-28 14:11:42,null,22.79023904211278,22.30000556304265,-4.701573820781195,0.6455823575938596,7.767282683093772,-1655,0.16862222615177486,tx2025zqwqjm,3.255311E7
106,2121191,121191,10142010000121191,114976,42-173-37618,BERRY SWD,GOODNIGHT MIDSTREAM,Water Management,GOODNIGHT MIDSTREAM,Water Management,BERRY SWD 1,1,TX,Permian - Midland,Glasscock,08,null,Saltwater Disposal,Active - Liquid,Injection Into Non-Productive Zone,40000.0,5600.0,0.0,null,Commercial,11300.0,14000.0,Class 2 - Injection of fluids associated with oil and natural gas production,Not in Permitting Process,Permian Deep,null,13.0,false,null,null,null,null,null,null,null,null,null,null,null,null,null,Permian Deep,Completed,2017-08-24,12800.0,12755.0,Vertical,true,2605.0,31.98046,-101.72665,Lat/Long from Well Surface Location,1510' FSL & 1779' FEL,2024-02-16 20:41:47,2025-11-28 14:11:42,null,25.237990008241816,24.952210449073146,-3.7872645916342678,0.6895047865454368,8.350444900752908,-1600,0.19234729462470046,tx2025zqwqjm,3.2942191E7
109,2098949,98949,10142010000098949,102275,42-173-31973,L. M. HARRISON ET AL,AQUA TERRA PERMIAN,Water Management,AQUA TERRA PERMIAN,Water Management,L. M. HARRISON ET AL 1W,1W,TX,Permian - Midland,Glasscock,08,null,Saltwater Disposal,Active - Liquid,Injection Into Non-Productive Zone,20000.0,2100.0,0.0,null,Commercial,4002.0,4196.0,Class 2 - Injection of fluids associated with oil and natural gas production,Not in Permitting Process,Permian Shallow,null,56.0,true,null,null,null,null,null,null,null,null,null,null,null,null,null,Permian Shallow & Deep,Completed,2010-11-03,8875.0,8875.0,Vertical,false,2712.0,31.844986,-101.700584,Lat/Long from Well Surface Location,421'FNL & 1568'FEL,2024-02-16 20:41:47,2025-11-28 14:11:42,null,25.

In [0]:

scenarioTSRDF,dPTimeSeriesR,wellIDsR,dayVecR = gist.runPressureScenariosTimeSeriesConv(eqDict,winWellsDF,winInjDF,verbose=verb)

/Workspace/Users/bill.curry@exxonmobil.com/GIST/lib/gistMC.py:3017: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  allScenariosDF=pd.concat([allScenariosDF,scenarioDF],ignore_index=True)


In [0]:
totalPPQuantilesDF=gi.prepTotalPressureTimeSeriesQuantilesPlot(dPTimeSeriesR,dayVecR,nQuantiles=21,epoch=pd.to_datetime('1970-01-01'),verbose=1)
totalPPSpaghettiDF=gi.prepTotalPressureTimeSeriesSpaghettiPlot(dPTimeSeriesR,dayVecR,gist.diffPPVec,epoch=pd.to_datetime('1970-01-01'),verbose=1)
print(totalPPQuantilesDF)

prepTotalPressureTimeSeriesPlot: deltaPP.shape= (4, 250, 585)  dayVec.shape= (585,)
prepTotalPressureTimeSeriesPlot: totalDeltaPP.shape= (250, 585)
prepTotalPressureTimeSeriesPlot: quantiles: [0.0, 4.8, 10.0, 14.9, 20.1, 24.9, 30.1, 34.9, 40.2, 45.0, 49.8, 55.0, 59.8, 65.1, 69.9, 75.1, 79.9, 85.1, 90.0, 95.2, 100.0]
prepTotalPressureTimeSeriesSpaghettiPlot: deltaPP.shape= (4, 250, 585)  dayVec.shape= (585,)
prepTotalPressureTimeSeriesSpaghettiPlot: totalDeltaPP.shape= (250, 585)
        DeltaPressure     Days  Realization  Percentile  Ordering       Date
7        5.248426e-16  14700.0            0        55.0     137.0 2010-04-01
15      -3.673899e-15  14780.0            0        14.9      37.0 2010-06-20
16      -2.175335e-15  14790.0            0        34.9      87.0 2010-06-30
17      -2.596590e-15  14800.0            0        34.9      87.0 2010-07-10
20       1.381165e-16  14830.0            0        65.1     162.0 2010-08-09
...               ...      ...          ...         ..

In [0]:
allPPQuantilesDF,allPPSpaghettiDF=gi.getPerWellPressureTimeSeriesSpaghettiAndQuantiles(dPTimeSeriesR,dayVecR,gist.diffPPVec,wellIDsR,nQuantiles=11,epoch=pd.to_datetime('01-01-1970'))

/Workspace/Users/bill.curry@exxonmobil.com/GIST/lib/gistMC.py:4036: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  PPQuantilesDF=pd.concat([PPQuantilesDF,winWellPPDF],ignore_index=True)
/Workspace/Users/bill.curry@exxonmobil.com/GIST/lib/gistMC.py:4037: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  PPSpaghettiDF=pd.concat([PPSpaghettiDF,wellPPDF[['DeltaPressure','Days','Realization','WellID','Diffusivity']]],ignore_index=True)


In [0]:
wellPressureDict=gi.prepPressureAndDisposalTimeSeriesPlots(allPPQuantilesDF,allPPSpaghettiDF,winWellsDF,winInjDF,orderedWellIDList[:-1],verbose=0)

###2.2.6: Sensitivity Analysis, Generate Tornado Plots

In [0]:
# Calculate sensitivities to different parameters
sensitivityDF,sensitivitySumDF = gist.getPressureSensitivity(winInjDF,winWellsDF,eqDict,verbose=1)
# I'm not getting negative pressures anymore
print(sensitivitySumDF)

        EventID  EventLatitude  ...                 Parameter  MedianPressure
0  tx2025zqwqjm      31.946411  ...                   Density        4.231814
3  tx2025zqwqjm      31.946411  ...        Interval Thickness        4.231814
6  tx2025zqwqjm      31.946411  ...              Permeability        4.231814
2  tx2025zqwqjm      31.946411  ...                  Porosity        4.231814
1  tx2025zqwqjm      31.946411  ...                 Viscosity        4.231814
4  tx2025zqwqjm      31.946411  ...  Vertical Compressibility        4.231814
5  tx2025zqwqjm      31.946411  ...     Fluid Compressibility        4.231814

[7 rows x 11 columns]


/Workspace/Users/bill.curry@exxonmobil.com/GIST/lib/gistMC.py:3017: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  allScenariosDF=pd.concat([allScenariosDF,scenarioDF],ignore_index=True)
/Workspace/Users/bill.curry@exxonmobil.com/GIST/lib/gistMC.py:2909: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sensitivityDF=pd.concat([sensitivityDF,wellDF],ignore_index=True)


In [0]:
# This is the date to calcaulte THdPdT0
futureEQDict=eqDict.copy()
futureEQDict['Origin Date']=pd.to_datetime('2026-9-1')
print(eqDict,futureEQDict)
rateDF = gist.getTHdPdT0(winWellsDF,winInjDF,eqDict,futureEQ,2)

{'SeismicEventId': 2889501, 'DataSource': 'TexNet Earthquake Catalog', 'DataSourceUrl': None, 'EventID': 'tx2025zqwqjm', 'EventTimeUtc': '2025-12-30T07:03:26Z', 'EventTimeInLocalTimeZone': '2025-12-30T01:03:26Z', 'EventTimeZone': 'CST', 'EventType': 'Earthquake', 'DepthKm': 8.837891, 'DepthErrorKm': 1.0284786, 'Magnitude': 2.5188963, 'MagnitudeError': nan, 'MagnitudeType': 'ml(texnet)', 'Location': 'Western Texas', 'Status': 'final', 'Latitude': 31.946411, 'LatitudeError': 0.22928765, 'Longitude': -101.46223, 'LongitudeError': 0.22001718, 'UpdatedDateUtc': '2025-12-30T08:36:09Z', 'StationCount': 30, 'RMS': 0.08108611, 'RMS_p': None, 'RMS_s': None, 'FocalMechanismAzimuthalGap': nan, 'County': 'Glasscock (TX)', 'State': 'Texas', 'DuplicateSetId': 18407, 'RankWithinDuplicateSet': 1, 'B3RecordAddedUTCDateTime': Timestamp('2025-12-30 10:57:47'), 'B3RecordUpdatedUTCDateTime': Timestamp('2026-01-07 10:46:47'), 'B3RecordDeletedUTCDateTime': NaT, 'Origin Date': datetime.date(2025, 12, 30), 'Ori

/Workspace/Users/bill.curry@exxonmobil.com/GIST/lib/gistMC.py:2770: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  disposalScenarios=pd.concat([disposalScenarios,scenarioDF],ignore_index=True)


###2.2.7: Output CSV files

In [0]:
gist.writeRealizations(initialRunIntervalPath+'PorePressureRealizations.csv')

In [0]:
scenarioDF.to_csv(initialRunIntervalPath+'scenarios.csv')

In [0]:


filteredDF.to_csv(initialRunIntervalPath+'filteredScenarios.csv')
pd.Series(data=orderedWellIDList).to_csv(initialRunIntervalPath+'wellIDOrder.csv')

In [0]:
mergedWellsDF.to_csv(initialRunIntervalPath+'RTwells.csv')
rtDF.to_csv(initialRunIntervalPath+'RTDF.csv')

In [0]:
disaggregationDF.to_csv(initialRunIntervalPath+'disaggregation.csv')

In [0]:
totalPPQuantilesDF.to_csv(initialRunIntervalPath+'totalPPQuantiles.csv')
totalPPSpaghettiDF.to_csv(initialRunIntervalPath+'totalPPSpaghetti.csv')

In [0]:
i=0
for wellDictKey, wellDictValue in wellPressureDict.items():
  wellID=wellDictValue['WellInfo']['ID'].to_list()[0]
  wellFilePrefix='/perWell/well_'+str(i)+'_'
  wellDictValue['PPQuantiles'].to_csv(initialRunIntervalPath+wellFilePrefix+'PPQuantiles.csv')
  wellDictValue['Disposal'].to_csv(initialRunIntervalPath+wellFilePrefix+'Disposal.csv')
  wellDictValue['Spaghetti'].to_csv(initialRunIntervalPath+wellFilePrefix+'Spaghetti.csv')
  wellDictValue['WellInfo'].to_csv(initialRunIntervalPath+wellFilePrefix+'WellInfo.csv')
  print('well',i,', ID:',wellID,' completed')
  i=i+1

well 0 , ID: 2105142  completed
well 1 , ID: 2098949  completed
well 2 , ID: 2111690  completed
well 3 , ID: 2121191  completed


In [0]:
scenarioTSRDF.to_csv(initialRunIntervalPath+'materialScenariosR.csv')
np.savez_compressed(initialRunIntervalPath+'timeSeriesR.npz', deltaPP=dPTimeSeriesR,dayVec=dayVecR,wellIDs=wellIDsR)

In [0]:
sensitivityDF.to_csv(initialRunIntervalPath+'sensitivity.csv')
sensitivitySumDF.to_csv(initialRunIntervalPath+'sensitivitySum.csv')

In [0]:
rateDF.to_csv(initialRunIntervalPath+'rates.csv')

#3. Correct Data

##3.1 Export Disposal Data

In [0]:
selectedWellsDF.to_csv(disposalPath+'selectedWells.csv')
ignoredWellsDF.to_csv(disposalPath+'ignoredWells.csv')
allWellsDF=pd.concat([selectedWellsDF,ignoredWellsDF])
allWellsDF.to_csv(disposalPath+'allInZoneWells.csv')
injDF.to_csv(disposalPath+'inj.csv')
injDF.to_csv(disposalPath+'inj.zip',compression='zip')

In [0]:
# End Year of time series forecast - I don't think that I need this now.
#EndYear=2030
# Rerun select wells with an updated earthquake time set to the last day of EndYear
#forecastEQ=eq.copy()
#forecastEQ['Origin Date']=pd.to_datetime('12-31-'+str(EndYear))
#selectedForecastWellsDF,ignoredForecastWellsDF,injDF=gist.findWells(forecastEQ,PE=False,verbose=verb)
##selectedWellsDF.to_csv(disposalPath+'forecastSelectedWells.csv')
#ignoredWellsDF.to_csv(disposalPath+'forecastIgnoredWells.csv')
#allWellsDF=pd.concat([selectedWellsDF,ignoredWellsDF])
#allWellsDF.to_csv(disposalPath+'forecastAllInZoneWells.csv')
#injDF.to_csv(disposalPath+'forecastPastInj.csv')

##3.2 Import Updated Disposal Data

In [0]:
# I need to merge the updated disposal information with the prior stuff
# Load updated injection file
# Well comparisons
# Get list of well IDs in the updated injection file
# Get list of well IDs in updated well file
# Check to see overlap in well IDs
#    updated vs. prior selected wells
#    updated vs. prior ignored wells
# FilteredIgnored =  prior ignored wells - (wells in updated file and prior ignored wells)
# Merge FilteredIgnored with with updated wells
# Load injection
#    Check for time sampling of injection - all wells must be the same
#    Get overall time vector - convert to days
#    Loop over wells in well file
#      Get all disposal data for well ID
#      Convert date values to days
#      
#mergedInjFile=